Vector Stores and retrievers

Here we familier with Langchain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data -- from(vertor) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference,as in the case of retrieval-augmented generation.

we will cover
1. Documents
2. Vector Stores
3. Retrievers

Langchain Documents has two important attributes 
1. Page_content: a String representing a content;
2. Metadata: a dict containing arbitrary metadata. The metadata attribute can capture information about the source of the document,its relationship to other documents,and other information.

In [1]:
from langchain_core.documents import Document

documents=[
    Document(
        page_content="Dogs are great companions,known for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source":"mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring reletively simple care.",
        metadata={"source":"fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source":"bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source":"mammal-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions,known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring reletively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]= os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key,model="openai/gpt-oss-20b")


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\ayush gupta\OneDrive\Desktop\LANGCHAIN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ayush gupta\OneDrive\Desktop\LANGCHAIN\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ayush gupta\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or t

In [6]:
## VectorStore 
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(documents,embedding=embeddings)


In [9]:
# Async query
await vectorstore.asimilarity_search("cat")

[Document(id='c0869aad-bd24-41eb-b3cd-66b7eb88642b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='385f573c-e0cb-4930-92ef-b3b46a786094', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions,known for their loyalty and friendliness.'),
 Document(id='73bc6087-3259-4b65-9657-8707173af571', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='a41ea063-f586-4473-86e9-d15da0fccd16', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [11]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='c0869aad-bd24-41eb-b3cd-66b7eb88642b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056814193726),
 (Document(id='385f573c-e0cb-4930-92ef-b3b46a786094', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions,known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='73bc6087-3259-4b65-9657-8707173af571', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956902503967285),
 (Document(id='a41ea063-f586-4473-86e9-d15da0fccd16', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.6657923460006714)]

Retrivers

Langchain vectorStore object do not subclass Runnable,and so connot immediately be integrated into Langchain expression Language chains.

Langchain Retrievers are Runnables, so they implement a standard set of methods(e.g., synchonous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

In [12]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='c0869aad-bd24-41eb-b3cd-66b7eb88642b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='385f573c-e0cb-4930-92ef-b3b46a786094', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions,known for their loyalty and friendliness.')]]

2nd method 
VectorSotres implement an as_retriever method that will generate a Retriever,Specifically a vectorStoreRetriever.These retrievers include specific search_type and search_kwargs attributs that identify what methods of the underlying vector store to call,and how to parameterize them. 

In [13]:
vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='c0869aad-bd24-41eb-b3cd-66b7eb88642b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='385f573c-e0cb-4930-92ef-b3b46a786094', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions,known for their loyalty and friendliness.')]]

In [14]:
## RAG --> retrieval augumented generative
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer this question using the provided content only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("tell me about dogs")
print(response.content)


Dogs are great companions, known for their loyalty and friendliness.
